<table style="width:100%; border-collapse:collapse; border-spacing:0; border:none; border-bottom:2px solid #e0e0e0; margin-bottom:16px; padding:0; margin-top:0;">
  <tr style="border:none; padding:0; margin:0;">
    <td style="width:144px; padding:16px 24px 16px 0; vertical-align:middle; border:none; outline:none; box-shadow:none;">
      <img src="https://github.com/TUIlmenauAMS/fs-ws2627-pinn-guitar/blob/main/assets/ams_group_tui.png?raw=1" width="120" style="border-radius:12px; display:block; margin:0;"/>
    </td>
    <td style="padding:16px 0 16px 0; vertical-align:middle; border:none; outline:none; box-shadow:none;">
      <h1 style="margin:0 0 4px 0; padding:0;">Forschungsseminar – Medieningenieurwissenschaften</h1>
      <h2 style="margin:0 0 8px 0; padding:0;">NB01: PDE Basics</h2>
      <p style="margin:0; padding:0;">
        <b>Institut für Medientechnik – Technische Universität Ilmenau</b><br>
        <a href="https://www.tu-ilmenau.de/universitaet/fakultaeten/fakultaet-elektrotechnik-und-informationstechnik/profil/institute-und-fachgebiete/fachgebiet-angewandte-mediensysteme" target="_blank">Institut für Medientechnik – AMS Group</a>
      </p>
      <p style="margin:4px 0 0 0; padding:0;">
        <b>Fachgebietsleiter:</b> Prof. Dr.-Ing. Gerald Schuller<br>
        <b>Wissenschaftlicher Mitarbeiter:</b> Renato Fera P. <a href="https://www.linkedin.com/in/renato-profeta/" target="_blank">LinkedIn</a>
      </p>
    </td>
  </tr>
</table>

In [ ]:
# ── Environment detection ──────────────────────────────────────────────────────
# Runs automatically. Installs packages on Colab; confirms venv on local/Codespaces.
import sys, subprocess

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _in_colab():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "numpy", "matplotlib", "sympy"])
    print("✓ Colab — packages installed via pip.")
else:
    # Local VS Code or GitHub Codespaces:
    # packages are managed by uv (pyproject.toml). Run 'uv sync' once in the
    # repo root, then select the .venv interpreter as the kernel in VS Code.
    print("✓ Local / Codespaces — using uv-managed venv.")
# ──────────────────────────────────────────────────────────────────────────────

# NB01: PDE Basics

> **After this notebook you will be able to:**
> - Explain what a PDE is and how it differs from an ODE
> - Identify boundary conditions (Dirichlet, Neumann) and initial conditions
> - Write down the 1D heat equation and explain what each term means physically
> - Derive the analytical solution step by step using separation of variables
> - Understand why a general series solution collapses to a single mode for our initial condition
> - Visualise PDE solutions as heatmaps and line plots in Python
> - Use SymPy to display and verify mathematical equations symbolically

In [ ]:
# ── SymPy setup — symbolic variables used throughout this notebook ─────────────
import sympy as sp
from sympy import (symbols, Function, Eq, sin, cos, exp, pi,
                   diff, simplify, factor, dsolve, oo, Sum, Integer)
from IPython.display import display

# Independent variables
x, t = sp.symbols('x t', real=True)

# Physical parameters (all positive)
L, alpha, c, k = sp.symbols('L alpha c k', positive=True)

# Separation-of-variables helpers
lam   = sp.Symbol('lambda', positive=True)   # eigenvalue λ > 0
n     = sp.Symbol('n', positive=True, integer=True)
B_n   = sp.Symbol('B_n')                     # Fourier coefficient

# u is the unknown function of x and t (used for PDEs)
u = sp.Function('u')

# X, T are the separated spatial and temporal factors
X = sp.Function('X')
T_func = sp.Function('T')

# y is a function of t only (used for the ODE example)
y = sp.Function('y')

# Enable LaTeX rendering in notebook output
sp.init_printing(use_latex='mathjax')

print("✓ SymPy symbolic variables defined. All subsequent cells can use them.")
# ──────────────────────────────────────────────────────────────────────────────

---

## 1. What is a PDE?

A **PDE (Partial Differential Equation)** is an equation that relates a function of *several* variables to its partial derivatives. The function we are trying to find is unknown — the PDE is the rule it must obey everywhere in space and time.

### 1.1 Contrast with an ODE

An **ODE (Ordinary Differential Equation)** involves a function of only *one* independent variable. A helpful analogy:

- An **ODE** tracks how one quantity (e.g. a population) changes over time — one axis.
- A **PDE** tracks how a quantity changes across both space *and* time simultaneously — a whole surface of values.

### 1.2 Examples

| Equation | Type | Physical meaning |
|----------|------|------------------|
| $\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$ | PDE — parabolic | Heat diffusion along a rod |
| $\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}$ | PDE — hyperbolic | Wave propagation on a string |
| $\frac{dy}{dt} = -k\,y$ | ODE | Radioactive decay |

### 1.3 Bridge to the project

The stiff guitar string in the project is governed by a **hyperbolic PDE** (wave-like), with an extra fourth-order term that makes the string stiff. The 1D heat equation is simpler (parabolic, second-order) — we use it as the teaching example in this notebook so you can focus on the *structure* of PDEs.

In [ ]:
# ── The three equations from the table, written as SymPy objects ───────────────

# 1. Heat equation: ∂u/∂t = α ∂²u/∂x²
heat_eq = Eq(u(x, t).diff(t), alpha * u(x, t).diff(x, 2))
print("Heat equation (parabolic PDE):")
display(heat_eq)

# 2. Wave equation: ∂²u/∂t² = c² ∂²u/∂x²
wave_eq = Eq(u(x, t).diff(t, 2), c**2 * u(x, t).diff(x, 2))
print("Wave equation (hyperbolic PDE):")
display(wave_eq)

# 3. ODE: dy/dt = -k·y
ode_eq = Eq(y(t).diff(t), -k * y(t))
print("Radioactive decay ODE:")
display(ode_eq)
# ──────────────────────────────────────────────────────────────────────────────

---

## 2. The 1D heat equation

**Physical setup:** Imagine a thin metal rod of length $L$, insulated along its sides so heat can only flow left and right. We want to know the temperature $u(x, t)$ at every position $x \in [0, L]$ and every time $t \geq 0$.

**The PDE:**

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

### Symbol table

| Symbol | Meaning | Typical unit |
|--------|---------|-------------|
| $u(x,t)$ | Temperature at position $x$, time $t$ | °C |
| $x$ | Position along the rod | m |
| $t$ | Time | s |
| $\alpha$ | Thermal diffusivity (material constant) | m²/s |
| $L$ | Length of the rod | m |

### Term-by-term intuition

- **Left side** $\partial u / \partial t$: how fast the temperature at a point is changing in time — positive means heating up, negative means cooling down.
- **Right side** $\alpha \, \partial^2 u / \partial x^2$: the *curvature* of the temperature profile in space. If a point is hotter than its neighbours (a peak), $\partial^2 u / \partial x^2 < 0$, so the temperature there decreases — the heat spreads outward. If a point is cooler (a valley), it heats up.

> **Key insight:** The PDE says: *the more curved the profile, the faster it flattens.*

**Connection to the guitar string:** In the project, $u(x,t)$ will be the transverse displacement of the string (not temperature), and the PDE will have additional terms — but the same intuition about spatial derivatives driving time evolution applies.

In [ ]:
# ── Heat equation as a SymPy object ───────────────────────────────────────────
print("The 1D heat equation:")
heat_eq = Eq(u(x, t).diff(t), alpha * u(x, t).diff(x, 2))
display(heat_eq)
# ──────────────────────────────────────────────────────────────────────────────

---

## 3. Boundary and initial conditions

**Why are conditions needed?** A PDE alone has infinitely many solutions. Conditions select the unique one that matches the physical setup. Analogy: $dy/dx = 2x$ has the solution family $y = x^2 + C$; specifying $y(0) = 3$ pins down $C = 3$ uniquely.

### Initial condition (IC)

Specifies the full state of the system at $t = 0$:

$$u(x,\,0) = f(x) \quad \text{for all } x \in [0, L]$$

This is the temperature profile along the rod at the moment we start observing it.

### Boundary conditions (BC)

Specify what happens at the spatial edges $x = 0$ and $x = L$ for all $t > 0$.

- **Dirichlet BC** — fix the *value* of $u$ at the boundary:
  $u(0,t) = 0$ and $u(L,t) = 0$. Physically: both ends are held in ice water (0 °C).

- **Neumann BC** — fix the *spatial derivative* of $u$ at the boundary:
  $\frac{\partial u}{\partial x}\big|_{x=0} = 0$. Physically: the left end is insulated (no heat flows out).

> **Fixed setup for this notebook:**
> - **Dirichlet BCs:** $u(0,t) = 0$ and $u(L,t) = 0$ — both ends held at zero.
> - **Sinusoidal IC:** $u(x,0) = \sin(\pi x / L)$ — a smooth arch that is already zero at both ends, so it is automatically consistent with the BCs.

Why is this IC convenient? Because $\sin(\pi \cdot 0 / L) = 0$ and $\sin(\pi \cdot L / L) = \sin(\pi) = 0$, the IC already satisfies the BCs at $t = 0$.

**Connection to the guitar string:** The stiff string project uses a similar pair of Dirichlet conditions ($u = 0$ at both ends) plus two additional conditions on the second spatial derivative ($\partial^2 u / \partial x^2 = 0$ at both ends, the simply-supported / pinned-end condition). Four conditions are needed because the string PDE is fourth-order in space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Initial condition: u(x, 0) = sin(πx/L) ───────────────────────────────────
u0 = sin(pi * x / L)
print("Initial condition:")
display(Eq(u(x, 0), u0))

# ── Dirichlet boundary conditions ─────────────────────────────────────────────
print("\nDirichlet boundary conditions:")
display(Eq(u(0, t), 0))
display(Eq(u(L, t), 0))

# ── Neumann boundary condition (shown for reference, not used in this problem) ─
print("\nNeumann BC (insulated left end — not used here, shown for reference):")
display(Eq(u(x, t).diff(x).subs(x, 0), 0))

# ── Verify: IC satisfies the Dirichlet BCs at t = 0 ──────────────────────────
val_left  = simplify(u0.subs(x, 0))    # u(0, 0)
val_right = simplify(u0.subs(x, L))    # u(L, 0)

print("\nConsistency check — does the IC satisfy the BCs?")
display(Eq(u(0, 0), val_left))
display(Eq(u(L, 0), val_right))

if val_left == 0 and val_right == 0:
    print("✓ The initial condition satisfies both Dirichlet BCs.")

# ── Grid setup and plot of the IC ─────────────────────────────────────────────
L_num = 1.0
N     = 200
x_num = np.linspace(0, L_num, N)
u0_num = np.sin(np.pi * x_num / L_num)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(x_num, u0_num, lw=2, color='steelblue')
axes[0].set(title=r'Initial condition  $u(x,0) = \sin(\pi x / L)$',
            xlabel='x  [m]', ylabel='u(x, 0)')
axes[0].axhline(0, color='k', lw=0.5)

axes[1].plot(x_num, u0_num, lw=2, color='steelblue', label=r'$u(x,0)$')
axes[1].scatter([0, L_num], [0, 0], color='red', zorder=5, s=80,
                label='Dirichlet BC: $u = 0$')
axes[1].set(title='Initial condition + boundary conditions',
            xlabel='x  [m]', ylabel='u')
axes[1].legend()

plt.tight_layout()
plt.show()
# ──────────────────────────────────────────────────────────────────────────────

---

## 4. Deriving the solution

We now derive the analytical solution to the heat equation step by step. The strategy is called **separation of variables**: we try to write the solution as a product of a purely spatial part and a purely temporal part, see where that leads, and then assemble the result.

The full problem we are solving is:

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}, \quad x \in [0, L],\; t > 0$$

with boundary conditions $u(0, t) = u(L, t) = 0$ and initial condition $u(x, 0) = \sin(\pi x / L)$.

### 4.1 The ansatz: separation of variables

We **propose** (this is an educated guess — an *ansatz*) that the solution can be written as a product:

$$u(x, t) = X(x) \cdot T(t)$$

where $X$ depends only on $x$ and $T$ depends only on $t$. Not every PDE admits this, but the heat equation does because it is linear and its coefficients do not mix $x$ and $t$.

Substituting into the PDE:

$$X(x)\, T'(t) = \alpha\, X''(x)\, T(t)$$

Dividing both sides by $X(x) T(t)$ (assuming both are non-zero):

$$\frac{T'(t)}{T(t)} = \alpha \frac{X''(x)}{X(x)}$$

**Key argument:** The left-hand side depends only on $t$; the right-hand side depends only on $x$. The only way a function of $t$ alone can equal a function of $x$ alone — for *all* $x$ and $t$ — is if both sides equal the same constant. We call that constant $-\lambda$:

$$\frac{T'(t)}{T(t)} = \alpha \frac{X''(x)}{X(x)} = -\lambda$$

The minus sign is a convention that will make the physics clearer shortly.

In [ ]:
# ── Step 1: write the ansatz u = X(x)·T(t) and substitute into the PDE ───────

# Represent the ansatz symbolically
# X(x) and T_func(t) are SymPy functions defined in the setup cell
ansatz = X(x) * T_func(t)
print("Ansatz  u(x, t) = X(x)·T(t):")
display(Eq(u(x, t), ansatz))

# ── Substitute into heat equation: ∂/∂t[X·T] = α ∂²/∂x²[X·T] ───────────────
lhs_sub = ansatz.diff(t)          # X(x)·T'(t)
rhs_sub = alpha * ansatz.diff(x, 2)  # α·X''(x)·T(t)

print("\nAfter substituting the ansatz into the PDE:")
display(Eq(lhs_sub, rhs_sub))

# ── Divide both sides by X(x)·T(t) ───────────────────────────────────────────
lhs_sep = simplify(lhs_sub / ansatz)   # T'(t)/T(t)
rhs_sep = simplify(rhs_sub / ansatz)   # α·X''(x)/X(x)

print("\nDivide both sides by X(x)·T(t)  →  each side depends on only one variable:")
print("  Left side  (function of t only):")
display(lhs_sep)
print("  Right side (function of x only, multiplied by α):")
display(rhs_sep)

print("\nBoth sides must equal the same constant, which we call  −λ  (λ > 0).")
print("This gives us two independent ODEs.")
# ──────────────────────────────────────────────────────────────────────────────

### 4.2 Two ODEs

Setting each side equal to $-\lambda$ yields one ODE for $X$ and one for $T$:

$$X''(x) + \lambda\, X(x) = 0$$
$$T'(t) + \alpha\,\lambda\, T(t) = 0$$

These are standard ODEs — much easier to solve than the original PDE.

In [ ]:
# ── Step 2: write out the two ODEs ────────────────────────────────────────────

# Spatial ODE: X'' + λ X = 0
spatial_ode = Eq(X(x).diff(x, 2) + lam * X(x), 0)
print("Spatial ODE:")
display(spatial_ode)

# Temporal ODE: T' + α λ T = 0
temporal_ode = Eq(T_func(t).diff(t) + alpha * lam * T_func(t), 0)
print("\nTemporal ODE:")
display(temporal_ode)
# ──────────────────────────────────────────────────────────────────────────────

### 4.3 Solving the spatial ODE

The spatial ODE is $X'' + \lambda X = 0$. Its general solution depends on the sign of $\lambda$:

- **$\lambda < 0$:** $X(x) = A e^{\sqrt{|\lambda|}\,x} + B e^{-\sqrt{|\lambda|}\,x}$ — exponential growth. The boundary conditions $X(0) = 0$ and $X(L) = 0$ force $A = B = 0$, giving only the trivial zero solution. Not useful.
- **$\lambda = 0$:** $X(x) = Ax + B$ — a straight line. The BCs again force $A = B = 0$. Not useful.
- **$\lambda > 0$:** $X(x) = A \sin(\sqrt{\lambda}\, x) + B \cos(\sqrt{\lambda}\, x)$ — oscillatory. This is the only case that can produce non-trivial solutions.

We work with $\lambda > 0$ and apply the boundary conditions:

1. $X(0) = 0 \Rightarrow B = 0$ (since $\cos(0) = 1 \neq 0$, we need $B = 0$).
2. $X(L) = 0 \Rightarrow A \sin(\sqrt{\lambda}\, L) = 0$. Since we want $A \neq 0$, we need $\sin(\sqrt{\lambda}\, L) = 0$, which means $\sqrt{\lambda}\, L = n\pi$ for integer $n = 1, 2, 3, \ldots$

This gives the **eigenvalues**:
$$\lambda_n = \left(\frac{n\pi}{L}\right)^2, \quad n = 1, 2, 3, \ldots$$

and the corresponding **eigenfunctions**:
$$X_n(x) = \sin\!\left(\frac{n\pi x}{L}\right)$$

Each integer $n$ gives a valid spatial mode. These are the natural "vibration modes" of the rod in space.

In [ ]:
# ── Step 3: solve the spatial ODE and apply boundary conditions ───────────────

# General solution of X'' + λ X = 0  (SymPy dsolve)
spatial_sol = dsolve(spatial_ode, X(x))
print("General solution of the spatial ODE:")
display(spatial_sol)

# ── Apply BC1: X(0) = 0  →  B = 0 ────────────────────────────────────────────
# From the general solution X = C1*sin(√λ x) + C2*cos(√λ x)
# X(0) = C2 = 0  →  only the sine term survives
print("\nApply BC  X(0) = 0:")
print("  → the cosine term vanishes; only  X(x) = C₁ sin(√λ x)  remains.")
X_reduced = sp.Symbol('C1') * sin(sp.sqrt(lam) * x)
display(Eq(X(x), X_reduced))

# ── Apply BC2: X(L) = 0  →  sin(√λ L) = 0 ───────────────────────────────────
print("\nApply BC  X(L) = 0:")
print("  C₁ · sin(√λ · L) = 0.  Since we want C₁ ≠ 0, we need:")
eigenvalue_condition = Eq(sp.sqrt(lam) * L, n * pi)
display(eigenvalue_condition)

# ── Eigenvalues λ_n ───────────────────────────────────────────────────────────
lambda_n = (n * pi / L)**2
print("\nEigenvalues λ_n = (nπ/L)²:")
display(Eq(sp.Symbol('lambda_n'), lambda_n))

# ── Eigenfunctions X_n ────────────────────────────────────────────────────────
X_n = sin(n * pi * x / L)
print("\nEigenfunctions (spatial modes)  X_n(x) = sin(nπx/L):")
display(Eq(X(x), X_n))

print("\nEach integer n = 1, 2, 3, … gives one valid spatial mode.")
# ──────────────────────────────────────────────────────────────────────────────

### 4.4 Solving the temporal ODE

With $\lambda_n = (n\pi/L)^2$ known, the temporal ODE becomes:

$$T'(t) + \alpha \left(\frac{n\pi}{L}\right)^2 T(t) = 0$$

This is a simple first-order linear ODE. Its solution is a decaying exponential:

$$T_n(t) = e^{-\alpha\,(n\pi/L)^2\, t}$$

The decay rate $\alpha (n\pi/L)^2$ grows with $n^2$: **higher spatial modes decay much faster** than lower ones. Mode $n = 2$ decays four times as fast as mode $n = 1$, mode $n = 3$ nine times as fast, and so on.

In [ ]:
# ── Step 4: solve the temporal ODE for each mode n ────────────────────────────

# Substitute eigenvalue λ_n into the temporal ODE
temporal_ode_n = Eq(T_func(t).diff(t) + alpha * lambda_n * T_func(t), 0)
print("Temporal ODE with λ_n substituted:")
display(temporal_ode_n)

# Solve with dsolve
temporal_sol = dsolve(temporal_ode_n, T_func(t))
print("\nGeneral solution of the temporal ODE:")
display(temporal_sol)

# Clean form (absorb constant into B_n later)
T_n = exp(-alpha * (n * pi / L)**2 * t)
print("\nTemporal factor (setting integration constant = 1, absorbed into B_n):")
display(Eq(sp.Function('T_n')(t), T_n))

print("\nNote: the exponent grows with n².")
print("Higher modes (larger n) decay exponentially faster.")
# ──────────────────────────────────────────────────────────────────────────────

### 4.5 The general series solution

Each pair $(X_n, T_n)$ satisfies the PDE and the boundary conditions. Because the heat equation is **linear**, any linear combination of solutions is also a solution. Therefore the **general solution** is an infinite sum over all modes:

$$u(x, t) = \sum_{n=1}^{\infty} B_n \sin\!\left(\frac{n\pi x}{L}\right) e^{-\alpha\,(n\pi/L)^2\, t}$$

The coefficients $B_n$ are free — they are determined by the initial condition. This is a **Fourier sine series**: each term is a sine wave in space that decays exponentially in time, and the full solution is a superposition of infinitely many such waves.

At $t = 0$, the exponentials all equal 1 and we get:

$$u(x, 0) = \sum_{n=1}^{\infty} B_n \sin\!\left(\frac{n\pi x}{L}\right) = f(x)$$

This is the Fourier sine series of the initial condition $f(x)$. In general, finding all $B_n$ requires computing integrals (Fourier coefficients). However, our specific initial condition makes this trivial, as we see next.

In [ ]:
# ── Step 5: assemble the general series solution ──────────────────────────────

# Display the general solution as an infinite sum
u_general = Sum(B_n * sin(n * pi * x / L) * exp(-alpha * (n * pi / L)**2 * t),
                (n, 1, oo))
print("General series solution:")
display(Eq(u(x, t), u_general))

# At t = 0, exponentials vanish → Fourier sine series of the IC
u_at_t0 = Sum(B_n * sin(n * pi * x / L), (n, 1, oo))
print("\nAt t = 0  (all exponentials = 1):")
display(Eq(u(x, 0), u_at_t0))

print("\nThis must equal the initial condition f(x) = sin(πx/L).")
print("Finding the B_n coefficients is the next step.")
# ──────────────────────────────────────────────────────────────────────────────

### 4.6 Why the series collapses to a single term

The general solution requires computing infinitely many Fourier coefficients $B_n$. But look at our initial condition:

$$f(x) = \sin\!\left(\frac{\pi x}{L}\right)$$

Compare this with the $n=1$ term of the series at $t=0$:

$$B_1 \sin\!\left(\frac{1 \cdot \pi x}{L}\right) = B_1 \sin\!\left(\frac{\pi x}{L}\right)$$

The initial condition **is exactly** the $n=1$ mode — it is already a pure sine wave with the right frequency. Therefore:

- $B_1 = 1$ (the coefficient of the $n=1$ mode matches perfectly)
- $B_n = 0$ for all $n \geq 2$ (no other mode is present in the IC)

The infinite series collapses to a single term. This is not a coincidence or a shortcut — it is a direct consequence of choosing an IC that happens to be one of the eigenfunctions of the problem.

> **Physical intuition:** Each mode $\sin(n\pi x/L)$ is an independent "shape" that the rod can adopt. If the rod starts in a pure shape (one mode only), it stays in that shape forever — it just decays in amplitude. The different modes do not mix under the heat equation.

The final solution is therefore:

$$\boxed{u(x, t) = \sin\!\left(\frac{\pi x}{L}\right) e^{-\alpha\,(\pi/L)^2\, t}}$$

In [ ]:
# ── Step 6: apply the IC to collapse the series to a single mode ──────────────

# Our IC is f(x) = sin(πx/L) — that is exactly the n=1 eigenfunction
print("Initial condition:")
display(Eq(u(x, 0), sin(pi * x / L)))

print("\nCompare with the n=1 term of the series at t=0:")
display(Eq(B_n, sp.Integer(1)))
print("  B₁ = 1  because sin(πx/L) already matches the n=1 mode exactly.")
print("  Bₙ = 0  for all n ≥ 2  (no higher modes present in the IC).")

# ── The exact solution ────────────────────────────────────────────────────────
u_exact = sin(pi * x / L) * exp(-alpha * (pi / L)**2 * t)

print("\n" + "="*60)
print("Exact analytical solution (series collapses to one term):")
print("="*60)
display(Eq(u(x, t), u_exact))
# ──────────────────────────────────────────────────────────────────────────────

### 4.7 Verification with SymPy

The final step is to check: does this solution actually satisfy the heat equation and the boundary and initial conditions? SymPy can do this symbolically, leaving no doubt.

In [ ]:
# ── Step 7: verify the solution satisfies the PDE and all conditions ──────────

print("── Verification 1: heat equation ────────────────────────────────────────")
lhs_verify = u_exact.diff(t)                    # ∂u/∂t
rhs_verify = alpha * u_exact.diff(x, 2)         # α ∂²u/∂x²
residual   = simplify(lhs_verify - rhs_verify)  # should be 0
print("PDE residual  (∂u/∂t  −  α ∂²u/∂x²):")
display(residual)
print("✓ Solution satisfies the heat equation." if residual == 0
      else "✗ Residual is not zero — check the solution.")

print("\n── Verification 2: boundary conditions ──────────────────────────────────")
bc_left  = simplify(u_exact.subs(x, 0))         # u(0, t) — should be 0
bc_right = simplify(u_exact.subs(x, L))         # u(L, t) — should be 0
display(Eq(u(0, t), bc_left))
display(Eq(u(L, t), bc_right))
print("✓ Both Dirichlet BCs satisfied." if bc_left == 0 and bc_right == 0
      else "✗ Boundary condition(s) not satisfied.")

print("\n── Verification 3: initial condition ────────────────────────────────────")
ic_check = simplify(u_exact.subs(t, 0) - sin(pi * x / L))   # should be 0
display(Eq(u(x, 0), u_exact.subs(t, 0)))
print("✓ Initial condition satisfied." if ic_check == 0
      else "✗ Initial condition not satisfied.")
# ──────────────────────────────────────────────────────────────────────────────

---

## 5. How to read a PDE solution: space–time plots

**The solution is a surface:** $u(x,t)$ assigns a value to every $(x, t)$ pair. It cannot be drawn as a single 2D curve; we need specialised visualisations.

### Two standard visualisations

1. **Snapshot line plots:** Choose a set of times $t_1 < t_2 < \ldots$ and plot $u(x, t_i)$ as a curve for each. Each curve is one horizontal "slice" through the solution surface. Easy to read exact values; good for comparing the profile at different moments.

2. **Space–time heatmap:** Place $x$ on the horizontal axis and $t$ on the vertical axis. Colour each pixel by the value of $u(x,t)$. Shows the entire solution at a glance — diffusion fronts, decay, oscillations are all immediately visible as patterns.

### How to read a heatmap axis by axis

- A horizontal *row* at height $t = t_0$ is the spatial profile at that instant.
- A vertical *column* at position $x = x_0$ is how the temperature at that point evolves over time.
- Bright (or warm-coloured) areas are high values; dark areas are low values.

In [ ]:
# ── Recall the exact solution we just derived ─────────────────────────────────
print("Exact solution (derived in Section 4):")
display(Eq(u(x, t), u_exact))

# ── The decay rate λ₁ = α(π/L)² ──────────────────────────────────────────────
lam1 = alpha * (pi / L)**2
print("\nDecay rate  λ₁ = α(π/L)²:")
display(Eq(sp.Symbol('lambda_1'), lam1))

print("\nThe amplitude decays as exp(−λ₁ t). Larger λ₁ → faster decay.")
print("At fixed L, increasing α (more diffusive material) makes the rod cool faster.")
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
alpha_num = 0.01    # thermal diffusivity [m²/s]
T_end     = 10.0    # end time for visualisation [s]
Nt        = 500     # time grid points

t_num = np.linspace(0, T_end, Nt)

# ── Exact solution on a 2D grid: U[i_time, i_space] ──────────────────────────
X_grid, T_grid = np.meshgrid(x_num, t_num)
U = np.sin(np.pi * X_grid / L_num) * np.exp(-alpha_num * (np.pi / L_num)**2 * T_grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel — snapshot line plots at selected times
snapshot_times = [0, 1, 2, 5, 10]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(snapshot_times)))
for t_snap, col in zip(snapshot_times, colors):
    idx = np.argmin(np.abs(t_num - t_snap))   # find nearest time index
    axes[0].plot(x_num, U[idx], color=col, lw=2, label=f't = {t_snap} s')
axes[0].set(title='Exact solution — snapshots at different times',
            xlabel='x  [m]', ylabel='u(x, t)')
axes[0].legend(fontsize=9)

# Right panel — space–time heatmap (x horizontal, t vertical)
im = axes[1].imshow(U, aspect='auto', origin='lower',
                    extent=[0, L_num, 0, T_end], cmap='hot')
plt.colorbar(im, ax=axes[1], label='u(x, t)')
axes[1].set(title='Exact solution — space–time heatmap',
            xlabel='x  [m]', ylabel='t  [s]')

plt.tight_layout()
plt.show()
# ──────────────────────────────────────────────────────────────────────────────

**What you should see:**
- **Snapshots (left):** The initial arch gradually shrinks and flattens. By $t = 10$ s the rod is nearly at uniform temperature (zero). The decay is smooth and exponential — exactly the $e^{-\alpha(\pi/L)^2 t}$ factor we derived.
- **Heatmap (right):** The top of the image (large $t$) is dark — the rod has cooled. The bottom (small $t$) is bright. The smooth arch at $t = 0$ is visible at the very bottom row. No oscillations — the heat equation only diffuses, it does not wave.

> The stiff guitar string solution will look very different: the heatmap will show travelling and reflecting waves, with high-frequency components decaying faster than low ones due to the stiffness and damping terms in the PDE.

---

## 6. Summary

**What you now know:**

1. A PDE is an equation involving a function of several variables and its partial derivatives. It governs how a quantity evolves everywhere in space and time.
2. A unique solution requires an **initial condition** (the state at $t = 0$) and **boundary conditions** (the behaviour at the spatial edges for all $t > 0$).
3. The **separation of variables** method rewrites $u(x,t) = X(x) T(t)$, turning one PDE into two independent ODEs — one in $x$ (yielding eigenfunctions $\sin(n\pi x/L)$), one in $t$ (yielding exponential decay).
4. The **general solution** is an infinite Fourier sine series. When the initial condition is itself one of the eigenfunctions, the series collapses to a single term — not by luck, but because the IC contains only one mode.
5. PDE solutions are visualised as **snapshot line plots** (one curve per time instant) or as **space–time heatmaps** (the full solution as a colour image).

**SymPy takeaway:** Every equation in this notebook was written both as LaTeX and as a live SymPy object — and SymPy verified the final solution by checking the PDE, the boundary conditions, and the initial condition symbolically.

**Connection to the stiff guitar string project:**
The stiff string PDE has the same two-variable structure $u(x,t)$, the same type of initial condition (a plucked profile), and the same type of Dirichlet boundary conditions ($u = 0$ at both ends). The differences are that it is a *wave* equation (second time derivative, not first), and it adds a fourth-order bending stiffness term and damping terms. The separation-of-variables logic, the eigenfunction structure, and the Fourier series assembly all carry over directly.

---

<div style="text-align:center; padding:18px 0 8px 0; margin-top:24px; border-top:2px solid #e0e0e0;">
  <p style="margin:0 0 6px 0;">
    <b>Forschungsseminar – Medieningenieurwissenschaften</b> &middot; NB01: PDE Basics<br>
    Institut für Medientechnik &ndash; Technische Universität Ilmenau
  </p>
  <p style="margin:0; font-size:0.95em;">
    Notebook created by <b>Renato Fera P.</b> &middot;
    <a href="https://www.linkedin.com/in/renato-profeta/" target="_blank">LinkedIn</a>
  </p>
</div>